In [ ]:
from google.colab import drive # to use goolge drive as disk

In [ ]:
drive.mount('/content/gdrive/') # mounting gdrive

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/gdrive/


In [ ]:
%cd /content/gdrive/My\ Drive/Live\ Sessions/24_Feb_2020/SpeechToText 
!ls 

/content/gdrive/My Drive/Live Sessions/24_Feb_2020/SpeechToText
1.PNG  3.PNG  5.PNG  7.PNG  api-key.json   _ipynb_checkpoints  requirements.txt
2.PNG  4.PNG  6.PNG  8.PNG  genevieve.wav  parts


# Preparing environment for this speech to text api

- Create an environment with python3

In [ ]:
!cat requirements.txt # libraries required

google-api-python-client==1.6.4
httplib2==0.10.3
oauth2client==4.1.2
pyasn1==0.4.2
pyasn1-modules==0.2.1
rsa==3.4.2
six==1.12.0
SpeechRecognition==3.8.1
tqdm==4.19.5
uritemplate==3.0.0


In [ ]:
!pip install -r requirements.txt # installing libraries as needed.

     |████████████████████████████████| 61kB 1.9MB/s 
     |████████████████████████████████| 204kB 7.7MB/s 
     |████████████████████████████████| 102kB 9.8MB/s 
     |████████████████████████████████| 71kB 7.9MB/s 
     |████████████████████████████████| 61kB 7.0MB/s 
     |████████████████████████████████| 51kB 6.4MB/s 
     |████████████████████████████████| 32.8MB 114kB/s 
     |████████████████████████████████| 61kB 7.0MB/s 
  Created wheel for httplib2: filename=httplib2-0.10.3-cp36-none-any.whl size=83987 sha256=8f01170851bb62ec408bc4b7dfc086188f6a5e1d73eab224c68939444b56d172
  Stored in directory: /root/.cache/pip/wheels/d6/f4/9c/f4eab4c19c0bde393b00a1f83afe12cc469852ff3810cd6f6d
Successfully built httplib2
  Found existing installation: uritemplate 3.0.1
    Uninstalling uritemplate-3.0.1:
      Successfully uninstalled uritemplate-3.0.1
  Found existing installation: httplib2 0.11.3
    Uninstalling httplib2-0.11.3:
      Successfully uninstalled httplib2-0.11.3
  Found exi

- Divide the source into multiple parts

```ffmpeg -i genevieve.wav -f segment -segment_time 30 -c copy parts/out%09d.wav```

In [ ]:
!ffmpeg -i genevieve.wav -f segment -segment_time 30 -c copy parts/out%09d.wav

ffmpeg version 3.4.6-0ubuntu0.18.04.1 Copyright (c) 2000-2019 the FFmpeg developers
  built with gcc 7 (Ubuntu 7.3.0-16ubuntu3)
  configuration: --prefix=/usr --extra-version=0ubuntu0.18.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --enable-gpl --disable-stripping --enable-avresample --enable-avisynth --enable-gnutls --enable-ladspa --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librubberband --enable-librsvg --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvorbis --enable-libvpx --enable-libwavpack --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --ena

In [ ]:
# from SpeechRecognition package, import speech_recognition module
# https://github.com/Uberi/speech_recognition#readme
import speech_recognition as sr
sr

<module 'speech_recognition' from '/usr/local/lib/python3.6/dist-packages/speech_recognition/__init__.py'>

In [ ]:
import os
from tqdm import tqdm

from multiprocessing.dummy import Pool
pool = Pool(8) # Number of concurrent threads

In [ ]:
pool

- Read the file contents of api-key.json file

In [ ]:
with open("api-key.json") as f:
    GOOGLE_CLOUD_SPEECH_CREDENTIALS = f.read()

In [ ]:
GOOGLE_CLOUD_SPEECH_CREDENTIALS[:100]+'...."' # cannot show you our key, sorry!

'{\n  "type": "service_account",\n  "project_id": "nifty-buffer-236512",\n  "private_key_id": "ce1f8d90d...."'

In [ ]:
# get the recogniser from the speech_recognition api
r = sr.Recognizer()
r

In [ ]:
# get the list of all the files that we want to convert
files = sorted(os.listdir('parts/'))
files

['out000000000.wav', 'out000000001.wav', 'out000000002.wav']

- Fucntion to convert the audio to text.

In [ ]:
name = 'parts/'+files[0]
from IPython.display import Audio
Audio(name)

In [ ]:
# Load audio file
with sr.AudioFile(name) as source:
    audio = r.record(source)
audio

In [ ]:
# Transcribe audio file
text = r.recognize_google_cloud(
    audio,
    credentials_json=GOOGLE_CLOUD_SPEECH_CREDENTIALS
    )
text

'this Dynamic Workshop aims to provide up-to-date information on pharmacological approaches, issues, and treatment in the geriatric population to assist in preventing medication-related problems, appropriately and effectively managing medications and compliance. The concept of polypharmacy parentheses taking multiple types of drugs parentheses will also be discussed, as the '

In [ ]:
def transcribe(data):
    idx, file = data
    name = "parts/" + file
    print(name + " started")
    # Load audio file
    with sr.AudioFile(name) as source:
        audio = r.record(source)
    # Transcribe audio file
    text = r.recognize_google_cloud(audio, credentials_json=GOOGLE_CLOUD_SPEECH_CREDENTIALS)
    print(name + " done")
    return {
        "idx": idx,
        "text": text
    }

In [ ]:
# running them in parallel to make it faster
all_text = pool.map(transcribe, enumerate(files))
pool.close()
pool.join()

parts/out000000000.wav startedparts/out000000001.wav startedparts/out000000002.wav started


parts/out000000002.wav done
parts/out000000001.wav done
parts/out000000000.wav done


In [ ]:
all_text

[{'idx': 0,
  'text': 'this Dynamic Workshop aims to provide up-to-date information on pharmacological approaches, issues, and treatment in the geriatric population to assist in preventing medication-related problems, appropriately and effectively managing medications and compliance. The concept of polypharmacy parentheses taking multiple types of drugs parentheses will also be discussed, as the '},
 {'idx': 1,
  'text': 'is a common issue that can impact adverse side effects in the geriatric population. Participants will leave with a knowledge and considerations of common drug interactions and how to minimize the effects that limit function. Summit professional education is approved provider of continuing education. This course is offered for 6. '},
 {'idx': 2,
  'text': '. discourse contains a Content classified under the both the domain of occupational therapy and professional issues. '}]